In [0]:
base_path = "/Volumes/nyc/default/"
yellow_path = base_path + "yellowtaxi/"

In [0]:
temp_df = spark.read.csv("/Volumes/nyc/default/yellowtaxilola/train.csv", header=True, inferSchema=True)

temp_df.printSchema() 

temp_df.show(5)

In [0]:
from pyspark.sql.types import StringType, LongType, IntegerType, DateType, StructType, StructField, DoubleType, TimestampType

yellowSchema = StructType([
    StructField("id", StringType(), True), 
    StructField("vendorID", IntegerType(), True), 
    StructField("pickupDatetime", TimestampType(), True), 
    StructField("dropoffDatetime", TimestampType(), True), 
    StructField("passengerCount", IntegerType(), True), 
    StructField("pickupLongitude", DoubleType(), True), 
    StructField("pickupLatitude", DoubleType(), True), 
    StructField("dropoffLongitude", DoubleType(), True), 
    StructField("dropoffLatitude", DoubleType(), True), 
    StructField("storeAndFwdFlag", StringType(), True), 
    StructField("tripDuration", IntegerType(), True)
])

yellow_path = "/Volumes/nyc/default/yellowtaxilola/train.csv"

yellow_taxi_kaggle_df = (
    spark.read
    .format("csv")
    .schema(yellowSchema)
    .option("header", "true")
    .load(yellow_path)
) 

In [0]:
from pyspark.sql.functions import expr 

a = yellow_taxi_kaggle_df.withColumn("YYYYMM", expr("year(dropoffDatetime)*100 + month(dropoffDatetime)")).groupBy("YYYYMM").count().orderBy("YYYYMM")

a.display()

In [0]:
%pip install h3

In [0]:
# 先取前 10 行，再交给 display 显示
display(yellow_taxi_kaggle_df.limit(10))

In [0]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType
import h3 

# 1. 注册 UDF
@udf(returnType=StringType())
def get_h3_spark(lat, lng):
    if lat is not None and lng is not None:
        return h3.latlng_to_cell(lat, lng, 8) # 这里固定分辨率为 8
    return None

# 2. 直接对千万级 Spark DataFrame 进行套用
enriched_kaggle_df = yellow_taxi_kaggle_df.withColumn(
    "pickup_h3", 
    get_h3_spark("pickupLatitude", "pickupLongitude")
)

enriched_kaggle_df.select("pickupLatitude", "pickupLongitude", "pickup_h3").show(5)

In [0]:
%pip install folium

In [0]:
import folium
from folium.plugins import HeatMap

# 1. 准备数据：确保字段名完全匹配（Kaggle 数据通常是 pickup_latitude/longitude）
# 加上过滤逻辑，防止空值破坏地图
plot_data = enriched_kaggle_df.select("pickupLatitude", "pickupLongitude") \
    .dropna() \
    .limit(2000) \
    .toPandas()

# 2. 创建地图：使用 'CartoDB positron'，因为它更柔和，能突出热力图的颜色
m = folium.Map(location=[40.7306, -73.9352], zoom_start=12, tiles='CartoDB positron')

# 3. 添加热力图
heat_values = plot_data[['pickupLatitude', 'pickupLongitude']].values.tolist()
HeatMap(heat_values, radius=10, blur=15).add_to(m)

# 4. 显示地图
m

In [0]:
import folium
from folium.plugins import HeatMap

# 1. 提高采样量 (增加到 10000 条，只要浏览器显存够，这会更精细)
plot_data_fine = enriched_kaggle_df.select("pickupLatitude", "pickupLongitude") \
    .dropna() \
    .limit(10000) \
    .toPandas()

# 2. 创建底图
m_fine = folium.Map(location=[40.7306, -73.9352], zoom_start=14, tiles='CartoDB positron')

# 3. 配置精细参数
# radius 调小（5-8），blur 调小（8-10），增加 min_opacity 确保孤立点也可见
heat_values_fine = plot_data_fine[['pickupLatitude', 'pickupLongitude']].values.tolist()
HeatMap(
    heat_values_fine, 
    radius=7, 
    blur=8, 
    min_opacity=0.4,
    gradient={0.4: 'blue', 0.65: 'lime', 1: 'red'} # 自定义渐变色，增强对比度
).add_to(m_fine)

m_fine

In [0]:
import folium
import h3
import pandas as pd
import numpy as np

# 1. 在 Spark 中进行聚合（这是 DE 的核心工作）
h3_counts = enriched_kaggle_df.groupBy("pickup_h3").count().toPandas()

# 2. 准备颜色映射 (使用分位数确保颜色分布均匀)
# 剔除异常值后，将 count 映射到 0-1 之间
max_count = h3_counts['count'].quantile(0.95)
h3_counts['score'] = h3_counts['count'] / max_count

# 3. 定义颜色函数
def get_color(score):
    # 从黄色(低)到红色(高)
    if score > 0.8: return '#800026' # 深红
    if score > 0.5: return '#BD0026'
    if score > 0.2: return '#FC4E2A'
    return '#FFEDA0' # 浅黄

# 4. 绘图
m_h3 = folium.Map(location=[40.7306, -73.9352], zoom_start=12, tiles='CartoDB dark_matter')

for _, row in h3_counts.head(1000).iterrows(): # 绘制前1000个最火爆的格子
    hex_id = row['pickup_h3']
    if hex_id:
        points = h3.cell_to_boundary(hex_id)
        folium.Polygon(
            locations=points,
            fill=True,
            fill_color=get_color(row['score']),
            color=None, # 不显示边框线，更美观
            fill_opacity=0.6,
            popup=f"H3_ID: {hex_id}<br>订单数: {row['count']}"
        ).add_to(m_h3)

m_h3